# Task 4 deep U-Net candidate
Attach FS2K.zip as a private Kaggle dataset (its contents should be extracted). No Task 2 bundle is needed. Run cells in order. Save full backup before ending session. First run a smoke check, then only five baseline epochs for inspection.

Uses the original splits, bilinear preprocessing and [0,1] metrics. Separate architecture/checkpoints: never restore baseline weights here. First inspect five epochs before extending. Both optimization phases use batch statistics; generator phase restores running buffers after backward. Start fresh; do not resume previous candidate weights.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path
import subprocess, sys
repo = Path('/kaggle/working/GenAI_A1')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AaizUrRehmanKazmi/GenAI_A1.git', str(repo)], check=True)
os.chdir(repo)
# For a NEW search, ensure these new files were pushed. For resume, keep the same revision.
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
assert (repo / 'training/train_classifier.py').is_file()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-optimization.txt'], check=True)


If installation requests a restart, restart and rerun setup. For later resume, keep the same source revision; do not blindly pull changed training sources.


In [ ]:
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Enable a GPU in notebook settings'
assert torch.cuda.device_count() == 1, 'Restart and run setup before importing torch'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


In [ ]:
candidates = [p.parent for p in Path('/kaggle/input').rglob('anno_train.json') if (p.parent/'photo').is_dir() and (p.parent/'sketch').is_dir()]
print(candidates)
assert len(candidates)==1, 'Attach one FS2K dataset or set raw to its exact root.'
raw = candidates[0]
subprocess.run([sys.executable,'-m','scripts.prepare_fs2k','--raw-dir',str(raw)],cwd=repo,check=True)


In [ ]:
smoke = Path('/kaggle/working/task4-candidate-bn-smoke')
cmd=[sys.executable,'-u','-m','training.train_task4_candidate_bn','--raw-dir',str(raw),'--output-dir',str(smoke),'--device','cuda','--train-limit','8','--val-per-style','1','--stop-after-epoch','1','--max-hours','.25']
if (smoke/'last.pt').exists():cmd+=['--resume',str(smoke/'last.pt')]
subprocess.run(cmd,cwd=repo,check=True)


In [ ]:
import shutil
output = Path('/kaggle/working/task4-candidate-bn')
restore_folder = None # after reset: attached complete baseline backup folder
if restore_folder is not None and not output.exists():
    source=Path(restore_folder)
    assert (source/'last.pt').is_file()
    shutil.copytree(source,output)
cmd=[sys.executable,'-u','-m','training.train_task4_candidate_bn','--raw-dir',str(raw),'--output-dir',str(output),'--device','cuda','--stop-after-epoch','5','--max-hours','2']
if (output/'last.pt').exists():cmd+=['--resume',str(output/'last.pt')]
subprocess.run(cmd,cwd=repo,check=True)


In [ ]:
import json
from IPython.display import display, FileLink, Image
history=json.loads((output/'history.json').read_text())
for row in history: print({k:v for k,v in row.items() if k!='validation'})
if (output/'preview.png').is_file():display(Image(filename=str(output/'preview.png')))
backup=shutil.make_archive('/kaggle/working/task4-candidate-bn_backup','zip',output.parent,output.name)
display(FileLink(backup))


Share five-epoch history and preview before continuing. Configuration targets 30 total epochs; stop-after-epoch pauses without changing that target. Resume must preserve data, source, config and visible GPU count. This is a provisional baseline, not the required Optuna search. Validation uses style-balanced L1 for checkpoint selection and also reports SSIM.
